## tl;dr

7モデルを同一採点定義で比較。総合首位はFable、時間・総トークン最少はTerra。以下はHTMLの数値と元スナップショットを独立照合する監査ノート。

## Context & Methods

2026年9月5日の各1実行。スコア・時間・トークンを混ぜず比較する。追加3モデルを新規採点し、既存4モデルは候補・採点器のハッシュ一致で前回値を継承。ログは全7件を再集計。

### Key Assumptions

本実行はAstra T4、Sol T2、Opus T1+T2、Fable T1、Terra T2、Luna T1、Sonnet T3。待機・準備・終了後依頼は除外。推論は出力の内数。料金は未測定。単一実行のため一般的能力順位ではない。

## Data

### 1. 監査済みスナップショットを読む

ベンチマークルートまたはこのノートのディレクトリから実行する。元のJSONLはプライベート監査で参照し、このノートには会話本文・推論本文を含めない。

In [1]:
import csv, json, math, sqlite3
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'evaluator/scoring.py').exists())
snap = root / 'analysis/final-review-20260905/expanded-7-models'
summary = json.loads((snap / 'combined_summary.json').read_text())
turns = list(csv.DictReader((snap / 'user_turns.csv').open(encoding='utf-8-sig')))
scores = {r['model']: json.loads((snap / (r['model'] + '_score.json')).read_text()) for r in summary}
print(f'{len(summary)} models; {len(turns)} user turns; {sum(t["selected_work_turn"] == "True" for t in turns)} work turns')


7 models; 21 user turns; 8 work turns


### 2. トークン・時間・点数・保全を照合する

In [2]:
for r in summary:
    work = [t for t in turns if t['model'] == r['model'] and t['selected_work_turn'] == 'True']
    assert sum(int(t['total_tokens']) for t in work) == r['total_tokens']
    assert math.isclose(sum(float(t['minutes']) for t in work), r['work_time_min'])
    assert r['input_tokens'] + r['output_tokens'] == r['total_tokens']
    assert math.isclose(sum(scores[r['model']]['category_scores'].values()), r['score'], abs_tol=.002)
    assert r['pytest_exit_code'] == 0
audit = json.loads((snap / 'evaluation_audit.json').read_text())
assert audit['original_candidates_unchanged'] and audit['completed_utc']
print('Token, time, category, pytest and immutable-snapshot checks: PASS')


Token, time, category, pytest and immutable-snapshot checks: PASS


## Results

### 3. 元データからHTMLのソースSQLを再実行する

SQL全文は同梱の `report_queries.sql` およびartifact内の出典に保存。

In [3]:
artifact = json.loads((root / 'evaluations/performance-report/artifact.json').read_text())
db = sqlite3.connect(':memory:')
db.row_factory = sqlite3.Row
db.execute('CREATE TABLE raw_evaluations (model TEXT, document TEXT)')
db.executemany('INSERT INTO raw_evaluations VALUES (?, ?)', [(m, json.dumps(d)) for m, d in scores.items()])
source = next(s for s in artifact['manifest']['sources'] if s['id'] == 'scores')
rows = list(db.execute(source['query']['sql']))
assert len(rows) == 49
for x in rows:
    assert math.isclose(x['category_score'], scores[x['model']]['category_scores'][x['category']])
print('49 category cells independently reconciled by SQL')


49 category cells independently reconciled by SQL


### 4. 7モデル比較（点 / 分 / 処理トークン）

In [4]:
for r in summary:
    print(f"{r['model']:7s} {r['score']:7.3f} / {r['work_time_min']:7.2f} min / {r['total_tokens']:12,} tokens")
print('Candidate-written pytest passes:', sum(r['verified_tests_passed'] for r in summary))


fable    94.473 /   55.94 min /   10,790,902 tokens
opus     93.431 /  124.22 min /   59,401,805 tokens
luna     91.021 /   32.99 min /   11,232,752 tokens
terra    82.835 /   18.30 min /    3,701,783 tokens
sol      80.438 /   34.60 min /    8,975,027 tokens
astra    80.103 /   51.97 min /    3,797,692 tokens
sonnet   75.050 /   62.97 min /   39,169,741 tokens
Candidate-written pytest passes: 403


## Takeaways

追加モデルで比較対象と用途別候補が変わる。Lunaは91.021点でAstra・Solを上回り、Terraは18.30分・3,701,783トークン。本実行と全セッション、キャッシュ読取とそれ以外を区別する。採点の形式依存・価格付け規約の留保はHTMLに保持した。

検証：標準ライブラリによる全コードセルの順次実行済み。nbformat/nbclientが環境にないためJupyterカーネル実行は使用していない。通常のNotebook v4.5形式として保存し、セル構造と実行結果を検査した。